In [ ]:
# Track ArUco markers

import cv2
import cv2.aruco as aruco
import numpy as np
import json

# Load the camera calibration parameters from './calibration_data.json'
with open('./calibration_data.json', 'r') as f:
    calibration_data = json.load(f)
    camera_matrix = np.array(calibration_data['camera_matrix'])
    dist_coeffs = np.array(calibration_data['distortion_coefficients'])

# Define the ArUco dictionary and parameters
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
parameters = aruco.DetectorParameters()

# Create detector
detector = aruco.ArucoDetector(aruco_dict, parameters)

inch_to_meter = 0.0254
marker_length = (5 + 12/16) * inch_to_meter 

obj_points = np.array([[-marker_length/2, marker_length/2, 0],
                        [marker_length/2, marker_length/2, 0],
                        [marker_length/2, -marker_length/2, 0],
                        [-marker_length/2, -marker_length/2, 0]], dtype=np.float32)

In [14]:
# Capture a frame from the camera and detect ArUco markers

import matplotlib.pyplot as plt

cap = cv2.VideoCapture(0)  # Use 0 for default camera
if not cap.isOpened():
    print("Cannot open camera")
    exit()

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture frame")
            break

        frame = cv2.undistort(frame, camera_matrix, dist_coeffs)

        # Detect markers
        corners, ids, rejected = detector.detectMarkers(frame)

        if ids is not None:
            ids = ids.flatten()
            print(f"Detected markers: {ids.flatten()}")

            aruco.drawDetectedMarkers(frame, corners, ids)

            # Marker ids 0, 1, 3 define world coordinates
            world_coordinates = {}
            for i in range(len(ids)):
                if ids[i] in [0, 1, 3]:
                    # Use solvePnP with all four corners of the marker
                    success, rvec, tvec = cv2.solvePnP(obj_points, corners[i], camera_matrix, dist_coeffs)
                    if success:
                        world_coordinates[ids[i]] = tvec.flatten()

                        # Project the first corner back to the image plane
                        projected_corner, _ = cv2.projectPoints(obj_points[0:1], rvec, tvec, camera_matrix, dist_coeffs)
                        corner = projected_corner[0][0]  # Extract x, y for drawing

                        cv2.circle(frame, tuple(corner.astype(int)), 5, (0, 255, 0), -1)
                        cv2.putText(frame, f"ID: {ids[i]}", tuple(corner.astype(int) + np.array([10, 10])), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

            # Check if the three markers make a right angle
            if all(marker in world_coordinates for marker in [0, 1, 3]):
                vector_01 = world_coordinates[1] - world_coordinates[0]  # Vector from marker 0 to marker 1
                vector_03 = world_coordinates[3] - world_coordinates[0]  # Vector from marker 0 to marker 3

                dot_product = np.dot(vector_01, vector_03)
                angle = np.degrees(np.arccos(dot_product / (np.linalg.norm(vector_01) * np.linalg.norm(vector_03))))

                if np.isclose(angle, 90, atol=5):  # Allow a tolerance of 5 degrees
                    print("Markers 0, 1, and 3 form a right angle.")
                else:
                    print(f"Markers 0, 1, and 3 do not form a right angle. Angle: {angle:.2f} degrees")

                # Define the world frame transformation
                origin = world_coordinates[0]
                x_axis = world_coordinates[3] - origin  # 0 to 3 is x-axis
                y_axis = world_coordinates[1] - origin  # 0 to 1 is y-axis

                # Normalize axes
                x_axis = x_axis / np.linalg.norm(x_axis)
                y_axis = y_axis / np.linalg.norm(y_axis)
                z_axis = np.cross(x_axis, y_axis)
                z_axis = z_axis / np.linalg.norm(z_axis)

                # Construct rotation matrix
                rotation_matrix = np.vstack([x_axis, y_axis, z_axis]).T
                translation_vector = origin

                # Create a 4x4 homogeneous transformation matrix
                world_to_camera_transform = np.eye(4)
                world_to_camera_transform[:3, :3] = rotation_matrix
                world_to_camera_transform[:3, 3] = translation_vector

                # print("World to camera transformation matrix:", world_to_camera_transform)

            # Draw axis for markers other than 0, 1, 3
            for i in range(len(ids)):
                if ids[i] not in [0, 1, 3]:
                    success, rvec, tvec = cv2.solvePnP(obj_points, corners[i], camera_matrix, dist_coeffs)
                    if success:
                        cv2.drawFrameAxes(frame, camera_matrix, dist_coeffs, rvec, tvec, 0.1)
 
                        # Transform rvec and tvec to world coordinates
                        camera_to_marker_transform = np.eye(4)
                        camera_to_marker_transform[:3, :3] = cv2.Rodrigues(rvec)[0]
                        camera_to_marker_transform[:3, 3] = tvec.flatten()

                        marker_in_world = np.dot(np.linalg.inv(world_to_camera_transform), camera_to_marker_transform)
                        # print(f"Marker {ids[i]} in world frame:", marker_in_world)

                        # print x, y, theta of the marker in world frame
                        x_world = marker_in_world[0, 3]
                        y_world = marker_in_world[1, 3]
                        theta_world = np.arctan2(marker_in_world[1, 0], marker_in_world[0, 0])
                        print(f"Marker {ids[i]} in world frame: x={x_world:.2f} m, y={y_world:.2f} m, theta={np.degrees(theta_world):.2f} degrees")

        cv2.imshow('frame', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
except KeyboardInterrupt:
    print("Exiting...")
finally:
    cap.release()
    cv2.destroyAllWindows()

Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right angle.
Marker 2 in world frame: x=0.00 m, y=0.61 m, theta=-89.21 degrees
Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right angle.
Marker 2 in world frame: x=0.00 m, y=0.61 m, theta=-89.27 degrees
Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right angle.
Marker 2 in world frame: x=0.00 m, y=0.61 m, theta=-89.21 degrees
Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right angle.
Marker 2 in world frame: x=0.00 m, y=0.61 m, theta=-89.22 degrees
Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right angle.
Marker 2 in world frame: x=0.00 m, y=0.61 m, theta=-89.22 degrees
Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right angle.
Marker 2 in world frame: x=0.00 m, y=0.61 m, theta=-89.46 degrees
Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right angle.
Marker 2 in world frame: x=0.00 m, y=0.61 m, theta=-89.25 degrees
Detected markers: [0 3 2 1]
Markers 0, 1, and 3 form a right a

In [ ]:
cap.release()
cv2.destroyAllWindows()